# Import Tools

In [ ]:
!pip install google-play-scraper pandas tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 2.3 MB/s eta 0:00:00


In [ ]:
!pip install emoji

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 36.8 MB/s eta 0:00:00


# Scraping Data Review Aplikasi Gojek

In [ ]:
from google_play_scraper import reviews_all
import pandas as pd

app_id = "com.gojek.app"

reviews = reviews_all(
    app_id,
    lang="id",       # bahasa Indonesia
    country="id"     # Play Store Indonesia
)

df = pd.DataFrame(reviews)
print("Total review berhasil diambil:", len(df))


Total review berhasil diambil: 1673024


In [ ]:
# Melakukan Penyimpanan data setelah di Scraping
from google.colab import drive
drive.mount('/content/drive')

df.to_csv("/content/drive/MyDrive/gojek_raw_review.csv", index=False)

Mounted at /content/drive


# Cleaning Data & Labeling

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/gojek_raw_review.csv")
print("Data awal:", len(df))

/tmp/ipython-input-563573960.py:3: DtypeWarning: Columns (8,9) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("/content/drive/MyDrive/gojek_raw_review.csv")


Data awal: 1673024


In [ ]:
df['score'].value_counts().sort_index()

,count
score,
0,1
1,250991
2,57105
3,85895
4,141145
5,1137887


In [ ]:
import re
import emoji

def clean_text(text):
    # handle None, NaN, dan non-string
    if not isinstance(text, str):
        return ""

    text = text.lower()

    # hapus URL
    text = re.sub(r"http\S+|www\S+", "", text)

    # hapus emoji
    text = emoji.replace_emoji(text, replace="")

    # hapus angka
    text = re.sub(r"\d+", "", text)

    # hapus simbol & tanda baca
    text = re.sub(r"[^a-z\s]", " ", text)

    # hapus spasi berlebih
    text = re.sub(r"\s+", " ", text)

    text = text.strip()

    # hapus kata < 3 huruf
    text = " ".join([w for w in text.split() if len(w) >= 3])

    return text

In [ ]:
df["clean_text"] = df["content"].apply(clean_text)

# buang teks kosong setelah cleaning
df = df[df["clean_text"].str.strip() != ""]

print("Setelah cleaning:", len(df))

Setelah cleaning: 1570755


In [ ]:
# buang duplikat teks
df = df.drop_duplicates(subset=["clean_text"])

print("Setelah filter kualitas:", len(df))

Setelah filter kualitas: 813436


In [ ]:
# Labeling
def label_sentiment(score):
    if score <= 2:
        return -1
    else:
        return 1

df["label"] = df["score"].apply(label_sentiment)

# Balancing Class

In [ ]:
print(df["label"].value_counts())

label
 1    1265293
-1     305462
Name: count, dtype: int64


In [ ]:
TARGET_PER_LABEL = 2000

df_balanced = (
    df.groupby('label', group_keys=False)
      .apply(lambda x: x.sample(
          n=min(len(x), TARGET_PER_LABEL),
          random_state=42
      ))
      .reset_index(drop=True)
)

/tmp/ipython-input-3950681516.py:5: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(


In [ ]:
df_balanced['label'].value_counts()

,count
label,
-1,2000
1,2000


# Dataset Final

In [ ]:
# Simpan hanya kolom yang dibutuhkan
df_final = df_balanced[['clean_text', 'label']]

df_final.to_csv("gojek_reviews_balanced.csv", index=False, encoding="utf-8")
print("Dataset sudah selesai")

Dataset sudah selesai
